In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders

from Models.FraudModel import FraudMLP
from Models.AutoEncoder import AutoEncoder

from Utils.TrainUtils import get_device, train_model

In [2]:
train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


In [3]:
DEVICE = get_device()
print(DEVICE)

autoencoder = AutoEncoder (
    input_dim=input_dim,
    latent_dim=16,              
    hidden_dims=[128, 64]
).to(DEVICE)

autoencoder.load_state_dict(torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE))

model = FraudMLP (
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.3,
    encoder=autoencoder,
    use_recon_error_vector=True,
    freeze_encoder=True
).to(DEVICE)

# So i don't repeat the boo boo
print(model.config)
print("Using encoder:", model.encoder is not None)
print("Freeze encoder:", model.freeze_encoder)
print("Use recon vector:", model.use_recon_error_vector)

cuda
{'input_dim': 776, 'hidden_dims': [1024, 512, 256, 128, 64], 'gated': True, 'dropout': 0.3, 'use_norm': True, 'has_encoder': True, 'freeze_encoder': True, 'encoder_latent_dim': 16, 'use_recon_error_vector': True}
Using encoder: True
Freeze encoder: True
Use recon vector: True


In [4]:
y = train_loader.dataset.y
pos_weight = (y == 0).sum().item() / (y == 1).sum().item()

print("Pos Weight:", pos_weight)

Pos Weight: 27.580278281911674


In [5]:
model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=130,
    lr=5e-5,
    device=DEVICE,
    pos_weight=pos_weight*0.5,
    save_path="GatedMLP_AE16_RECONVEC_V1",
    optimize_for="f2",
    num_thresholds=300,
)

[INFO] Saved best model

Epoch 1/130
Train Loss:      0.680147
Val Loss:        0.588878
Val @ 0.5 -> Acc: 0.9416 | Prec: 0.3115 | Rec: 0.5525 | F1: 0.3984 | F2: 0.4785
Best threshold:  0.5061
Best metrics -> Acc: 0.9430 | Prec: 0.3185 | Rec: 0.5506 | F1: 0.4035 | F2: 0.4805
Confusion @ best -> TP: 1138 | FP: 2435 | TN: 54552 | FN: 929
Epoch time:      14.08s
[INFO] Saved best model

Epoch 2/130
Train Loss:      0.608106
Val Loss:        0.558357
Val @ 0.5 -> Acc: 0.9319 | Prec: 0.2839 | Rec: 0.6212 | F1: 0.3897 | F2: 0.5019
Best threshold:  0.5459
Best metrics -> Acc: 0.9431 | Prec: 0.3271 | Rec: 0.5926 | F1: 0.4215 | F2: 0.5099
Confusion @ best -> TP: 1225 | FP: 2520 | TN: 54467 | FN: 842
Epoch time:      14.20s
[INFO] Saved best model

Epoch 3/130
Train Loss:      0.577091
Val Loss:        0.537871
Val @ 0.5 -> Acc: 0.9352 | Prec: 0.2983 | Rec: 0.6294 | F1: 0.4048 | F2: 0.5151
Best threshold:  0.6027
Best metrics -> Acc: 0.9523 | Prec: 0.3823 | Rec: 0.5868 | F1: 0.4630 | F2: 0.5301


In [6]:
# TESTING
import json

from Utils.TrainUtils import evaluate

DEVICE = get_device()

In [7]:
with open("checkpoints/GatedMLP_V6/summary.json", "r", encoding="utf-8") as f:
    summary_v2 = json.load(f)

threshold_v2 = summary_v2["best_val_metrics"]["best_threshold"]

model_v2 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=None,
    freeze_encoder=False,
).to(DEVICE)

ckpt_v2 = torch.load("checkpoints/GatedMLP_V6/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_v2:
    model_v2.load_state_dict(ckpt_v2["model_state_dict"])
else:
    model_v2.load_state_dict(ckpt_v2)

model_v2.eval()

criterion_v2 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_v2["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_v2 = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=threshold_v2,
    sweep_thresholds=False,
)

print("=== GatedMLP_V6 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2[k]}")

test_metrics_v2_fixed = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_V6 Test (0.5 Threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2_fixed[k]}")

=== GatedMLP_V6 Test ===
loss: 0.8150083881387592
threshold: 0.7119498252868652
accuracy: 0.9737020354250392
precision: 0.611667392248099
recall: 0.6800580832493706
f1: 0.6440522526319669
f2: 0.6651832191553039
tp: 1405
fp: 892
tn: 56096
fn: 661
=== GatedMLP_V6 Test (0.5 Threshold) ===
loss: 0.8150083881387592
threshold: 0.5
accuracy: 0.9636434449823952
precision: 0.4865044984988787
recall: 0.7066795740527266
f1: 0.5762778716234734
f2: 0.6480248533021254
tp: 1460
fp: 1541
tn: 55447
fn: 606


In [8]:
with open("checkpoints/GatedMLP_AE16_RECONVEC_V1/summary.json", "r", encoding="utf-8") as f:
    summary_ae16_reconvec = json.load(f)

threshold_ae16_reconvec = summary_ae16_reconvec["best_val_metrics"]["best_threshold"]

autoencoder = AutoEncoder(
    input_dim=input_dim,
    latent_dim=16,
    hidden_dims=[128, 64],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt = torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE)
if "model_state_dict" in ae_ckpt:
    autoencoder.load_state_dict(ae_ckpt["model_state_dict"])
else:
    autoencoder.load_state_dict(ae_ckpt)
autoencoder.eval()

model_ae16_reconvec = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=autoencoder,
    use_recon_error_vector=True,
    freeze_encoder=True,
).to(DEVICE)

ckpt_ae16 = torch.load("checkpoints/GatedMLP_AE16_RECONVEC_V1/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_ae16:
    model_ae16_reconvec.load_state_dict(ckpt_ae16["model_state_dict"])
else:
    model_ae16_reconvec.load_state_dict(ckpt_ae16)

model_ae16_reconvec.eval()

criterion_ae16 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_ae16_reconvec["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_ae16 = evaluate(
    model=model_ae16_reconvec,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=threshold_ae16_reconvec,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16_RECONVEC Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16[k]}")

test_metrics_ae16_fixed = evaluate(
    model=model_ae16_reconvec,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16_RECONVEC Test (0.5 threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16_fixed[k]}")

=== GatedMLP_AE16_RECONVEC Test ===
loss: 0.6460871352971375
threshold: 0.6552073955535889
accuracy: 0.971314390218957
precision: 0.5796232876687516
recall: 0.6553727008680766
f1: 0.6151749155067016
f2: 0.6386792431323666
tp: 1354
fp: 982
tn: 56006
fn: 712
=== GatedMLP_AE16_RECONVEC Test (0.5 threshold) ===
loss: 0.6460871352971375
threshold: 0.5
accuracy: 0.9593592305346024
precision: 0.4481688392289008
recall: 0.6989351403644776
f1: 0.5461422040114661
f2: 0.6285913260483482
tp: 1444
fp: 1778
tn: 55210
fn: 622
